# Lesson 2 solutions

In [ ]:
# discopt course compat shim — installs `add_variable / add_variables /
# add_constraint / add_constraints / is_convex` and a friendly `.solve(...)`
# on `discopt.Model`. Run this cell first.
import sys, pathlib
_repo = pathlib.Path.cwd()
while _repo != _repo.parent and not (_repo / "course").is_dir(): _repo = _repo.parent
if str(_repo) not in sys.path: sys.path.insert(0, str(_repo))
from course._compat import *  # noqa: F401,F403


In [ ]:
import numpy as np
# x1 free -> x1+ - x1-, slack on each inequality
# vars: [x1+, x1-, x2, s1, s2]
c = np.array([1.0, -1.0, -2.0, 0.0, 0.0])
A = np.array([
    [ 1, -1, 1, 1, 0],   # x1+ - x1- + x2 + s1 = 4
    [-1,  1, 1, 0, 1],   # -x1+ + x1- + x2 + s2 = 1   (after multiplying x1-x2 >= -1 -> -(x1-x2) <= 1)
])
b = np.array([4.0, 1.0])


In [ ]:
import numpy as np, discopt as do
m = do.Model(); x1 = m.add_variable(lb=-1e6, ub=1e6); x2 = m.add_variable(lb=0)
m.add_constraint(x1 + x2 <= 4); m.add_constraint(x1 - x2 >= -1)
m.minimize(x1 - 2*x2); r = m.solve()
# By inspection, x2 = 4, x1 = 0 -> obj = -8
print(r.objective, r.dual)


In [ ]:
import numpy as np, discopt as do
def km(n):
    m = do.Model(); x = m.add_variables(n, lb=0)
    for i in range(n):
        m.add_constraint(2*sum(10**(i-j) * x[j] for j in range(i)) + x[i] <= 100**i)
    m.maximize(sum(10**(n-1-j) * x[j] for j in range(n)))
    return m.solve()
for n in [3, 5, 7]: r = km(n); print(n, r.objective)


In [ ]:
# Cost change should equal dy * 50 (i.e., shadow price * 50)
# (numerical verification omitted for brevity)


Bounded: any feasible LP with bounded variables. Unbounded: $\min -x$ s.t. $x \ge 0$. Infeasible: $x \ge 1, x \le 0$. Solver detects these via primal/dual feasibility and the dual certificate (Farkas' lemma).